# 🎓 Student Career Success Prediction
### Complete Project — Backend (ML Pipeline) + Frontend (Streamlit UI)
**Dataset:** student_career_success_dataset.csv  
**Author:** YourName  
**Date:** 2025

---
## Table of Contents
1. [Project Overview](#1)
2. [Import Libraries](#2)
3. [Load & Explore Data (EDA)](#3)
4. [Data Preprocessing](#4)
5. [Feature Engineering](#5)
6. [Model Training & Evaluation](#6)
7. [Model Comparison & Selection](#7)
8. [Save Model Artifacts](#8)
9. [Streamlit Frontend App (Single File)](#9)

---
## 1. Project Overview <a id='1'></a>

This project predicts whether a student will be **Placed** or **Not Placed** based on academic,
technical, and soft-skill attributes. It also estimates **Starting Salary** for placed students.

**Target Variables:**
- `Placement_Status` → Binary Classification (Placed / Not Placed)
- `Starting_Salary_USD` → Regression (for placed students)

**Models Used:**
- Logistic Regression
- Random Forest Classifier
- Gradient Boosting Classifier (XGBoost style)
- Random Forest Regressor (salary prediction)

**Frontend:** Streamlit single-file app with interactive prediction form, EDA charts, and feature importance.

---
## 2. Import Libraries <a id='2'></a>

In [ ]:
# ── Core
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Visualisation
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import seaborn as sns

# ── Sklearn preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# ── Classification models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

# ── Regression models
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge

# ── Metrics
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    mean_absolute_error, mean_squared_error, r2_score
)

# ── Model persistence
import joblib
import os

print('All libraries imported successfully ✓')

---
## 3. Load & Explore Data (EDA) <a id='3'></a>

In [ ]:
df = pd.read_csv('student_career_success_dataset.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'No missing values ✓')

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#3b82d4', '#e74c3c']
counts = df['Placement_Status'].value_counts()
axes[0].bar(counts.index, counts.values, color=colors)
axes[0].set_title('Placement Status Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=10)

tier_counts = df[df['Placement_Status']=='Placed']['Company_Tier'].value_counts()
axes[1].pie(tier_counts.values, labels=tier_counts.index, autopct='%1.1f%%',
            colors=['#3b82d4','#7c5cd8','#27ae60'], startangle=90)
axes[1].set_title('Company Tier Distribution (Placed Students)', fontweight='bold')

plt.tight_layout()
plt.savefig('plot_target_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# CGPA vs Placement
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, grp in df.groupby('Placement_Status'):
    axes[0].hist(grp['CGPA'], bins=20, alpha=0.6, label=label)
axes[0].set_title('CGPA Distribution by Placement Status', fontweight='bold')
axes[0].set_xlabel('CGPA')
axes[0].legend()

sns.boxplot(data=df, x='Placement_Status', y='Employability_Score',
            palette=['#3b82d4','#e74c3c'], ax=axes[1])
axes[1].set_title('Employability Score vs Placement Status', fontweight='bold')

plt.tight_layout()
plt.savefig('plot_cgpa_employability.png', bbox_inches='tight')
plt.show()

In [ ]:
# Placement rate by Major
major_placement = df.groupby('Major')['Placement_Status'].apply(
    lambda x: (x=='Placed').sum() / len(x) * 100
).sort_values(ascending=False)

plt.figure(figsize=(10, 4))
bars = plt.bar(major_placement.index, major_placement.values, color='#3b82d4')
plt.title('Placement Rate by Major (%)', fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.ylabel('Placement Rate (%)')
for bar, val in zip(bars, major_placement.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('plot_major_placement.png', bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap (numeric features)
numeric_cols = df.select_dtypes(include=np.number).drop(columns=['Student_ID'], errors='ignore').columns
corr = df[numeric_cols].corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='Blues',
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix — Numeric Features', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('plot_correlation.png', bbox_inches='tight')
plt.show()

In [ ]:
# Salary distribution for placed students
placed = df[df['Placement_Status']=='Placed']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(placed['Starting_Salary_USD'], bins=40, color='#27ae60', edgecolor='white')
axes[0].set_title('Starting Salary Distribution (Placed)', fontweight='bold')
axes[0].set_xlabel('Salary (USD)')

tier_salary = placed.groupby('Company_Tier')['Starting_Salary_USD'].median().sort_values(ascending=False)
axes[1].bar(tier_salary.index, tier_salary.values, color=['#3b82d4','#7c5cd8','#27ae60'])
axes[1].set_title('Median Salary by Company Tier', fontweight='bold')
axes[1].set_ylabel('Median Salary (USD)')
for i, (label, val) in enumerate(tier_salary.items()):
    axes[1].text(i, val + 500, f'${val:,.0f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('plot_salary.png', bbox_inches='tight')
plt.show()

---
## 4. Data Preprocessing <a id='4'></a>

In [ ]:
# Drop irrelevant columns
drop_cols = ['Student_ID', 'Company_Tier', 'Career_Field', 'Placement_Mode']
df_model = df.drop(columns=drop_cols)

# Binary target
df_model['Target'] = (df_model['Placement_Status'] == 'Placed').astype(int)
df_model = df_model.drop(columns=['Placement_Status'])

# Salary target (regression — only placed students)
df_placed = df[df['Placement_Status'] == 'Placed'].copy()
df_placed = df_placed.drop(columns=['Student_ID', 'Placement_Status', 'Company_Tier',
                                     'Career_Field', 'Placement_Mode'])

print('Classification dataset shape:', df_model.shape)
print('Regression dataset shape (placed only):', df_placed.shape)

In [ ]:
# Define feature types
categorical_cols = ['Gender', 'University_Year', 'Major', 'Academic_Performance',
                    'GitHub_Profile', 'Leadership_Experience', 'LinkedIn_Profile',
                    'English_Proficiency']

numeric_cols_clf = [c for c in df_model.columns if c not in categorical_cols + ['Target']]

print('Numeric features:', numeric_cols_clf)
print('Categorical features:', categorical_cols)

In [ ]:
# Build preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols_clf),
    ('cat', categorical_transformer, categorical_cols)
])

print('Preprocessor built ✓')

---
## 5. Feature Engineering <a id='5'></a>

In [ ]:
# Engineered features
def add_engineered_features(data):
    d = data.copy()
    # Composite skill score
    d['Skill_Score'] = (
        d['Programming_Skill'] * 0.3 +
        d['Communication_Skills'] * 0.2 +
        d['Problem_Solving'] * 0.25 +
        d['Teamwork'] * 0.15 +
        d['Interview_Score'] * 0.1
    )
    # Activity index
    d['Activity_Index'] = d['Projects_Completed'] + d['Certifications'] + d['Hackathons']
    # Academic intensity
    d['Academic_Intensity'] = d['CGPA'] * d['Study_Hours_Per_Week'] / 10
    return d

df_model = add_engineered_features(df_model)
df_placed = add_engineered_features(df_placed)

# Update numeric cols
numeric_cols_clf = [c for c in df_model.columns if c not in categorical_cols + ['Target']]

print('Engineered features added ✓')
print('Total features:', len(numeric_cols_clf) + len(categorical_cols))

---
## 6. Model Training & Evaluation <a id='6'></a>
### 6a. Classification — Placement Status

In [ ]:
X_clf = df_model.drop(columns=['Target'])
y_clf = df_model['Target']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)
print(f'Train: {X_train_c.shape}, Test: {X_test_c.shape}')

In [ ]:
# Rebuild preprocessor with updated numeric cols
numeric_cols_clf_eng = [c for c in X_clf.columns if c not in categorical_cols]

preprocessor_clf = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols_clf_eng),
    ('cat', categorical_transformer, categorical_cols)
])

classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, random_state=42),
}

clf_results = {}

for name, clf in classifiers.items():
    pipe = Pipeline(steps=[('pre', preprocessor_clf), ('clf', clf)])
    pipe.fit(X_train_c, y_train_c)
    y_pred = pipe.predict(X_test_c)
    y_prob = pipe.predict_proba(X_test_c)[:, 1]
    acc = accuracy_score(y_test_c, y_pred)
    auc = roc_auc_score(y_test_c, y_prob)
    clf_results[name] = {'pipeline': pipe, 'accuracy': acc, 'auc': auc,
                          'y_pred': y_pred, 'y_prob': y_prob}
    print(f'{name:<25} Accuracy: {acc:.4f}   AUC: {auc:.4f}')

In [ ]:
# Best classifier
best_clf_name = max(clf_results, key=lambda k: clf_results[k]['auc'])
best_clf = clf_results[best_clf_name]
print(f'Best Classifier: {best_clf_name}')
print(classification_report(y_test_c, best_clf['y_pred'],
                             target_names=['Not Placed', 'Placed']))

In [ ]:
# Confusion matrix + ROC curve
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(y_test_c, best_clf['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Not Placed','Placed'],
            yticklabels=['Not Placed','Placed'])
axes[0].set_title(f'Confusion Matrix — {best_clf_name}', fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

fpr, tpr, _ = roc_curve(y_test_c, best_clf['y_prob'])
axes[1].plot(fpr, tpr, color='#3b82d4', lw=2,
             label=f'AUC = {best_clf["auc"]:.4f}')
axes[1].plot([0,1],[0,1],'k--', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title(f'ROC Curve — {best_clf_name}', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('plot_clf_evaluation.png', bbox_inches='tight')
plt.show()

### 6b. Regression — Starting Salary Prediction

In [ ]:
X_reg = df_placed.drop(columns=['Starting_Salary_USD'])
y_reg = df_placed['Starting_Salary_USD']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

cat_cols_reg = [c for c in categorical_cols if c in X_reg.columns]
num_cols_reg = [c for c in X_reg.columns if c not in cat_cols_reg]

preprocessor_reg = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols_reg),
    ('cat', categorical_transformer, cat_cols_reg)
])

regressors = {
    'Linear Regression':    LinearRegression(),
    'Ridge Regression':     Ridge(alpha=1.0),
    'Random Forest':        RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting':    GradientBoostingRegressor(n_estimators=200, random_state=42),
}

reg_results = {}

for name, reg in regressors.items():
    pipe = Pipeline(steps=[('pre', preprocessor_reg), ('reg', reg)])
    pipe.fit(X_train_r, y_train_r)
    y_pred = pipe.predict(X_test_r)
    mae  = mean_absolute_error(y_test_r, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_r, y_pred))
    r2   = r2_score(y_test_r, y_pred)
    reg_results[name] = {'pipeline': pipe, 'MAE': mae, 'RMSE': rmse, 'R2': r2,
                          'y_pred': y_pred}
    print(f'{name:<25} MAE: ${mae:,.0f}   RMSE: ${rmse:,.0f}   R²: {r2:.4f}')

In [ ]:
best_reg_name = max(reg_results, key=lambda k: reg_results[k]['R2'])
best_reg = reg_results[best_reg_name]
print(f'Best Regressor: {best_reg_name}  R²={best_reg["R2"]:.4f}')

plt.figure(figsize=(7, 5))
plt.scatter(y_test_r, best_reg['y_pred'], alpha=0.3, color='#3b82d4', s=10)
mn, mx = y_test_r.min(), y_test_r.max()
plt.plot([mn, mx], [mn, mx], 'r--', lw=1.5)
plt.xlabel('Actual Salary (USD)')
plt.ylabel('Predicted Salary (USD)')
plt.title(f'Actual vs Predicted Salary — {best_reg_name}', fontweight='bold')
plt.tight_layout()
plt.savefig('plot_salary_pred.png', bbox_inches='tight')
plt.show()

---
## 7. Model Comparison & Selection <a id='7'></a>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Classification comparison
names_c = list(clf_results.keys())
accs    = [clf_results[n]['accuracy'] for n in names_c]
aucs    = [clf_results[n]['auc'] for n in names_c]

x = np.arange(len(names_c))
w = 0.35
axes[0].bar(x - w/2, accs, w, label='Accuracy', color='#3b82d4')
axes[0].bar(x + w/2, aucs, w, label='AUC-ROC',  color='#7c5cd8')
axes[0].set_xticks(x)
axes[0].set_xticklabels(names_c, rotation=20, ha='right', fontsize=9)
axes[0].set_ylim(0.5, 1.05)
axes[0].set_title('Classifier Comparison', fontweight='bold')
axes[0].legend()

# Regression comparison
names_r = list(reg_results.keys())
r2s = [reg_results[n]['R2'] for n in names_r]
axes[1].bar(names_r, r2s, color=['#27ae60','#3b82d4','#7c5cd8','#e74c3c'])
axes[1].set_title('Regressor Comparison (R² Score)', fontweight='bold')
axes[1].set_xticklabels(names_r, rotation=20, ha='right', fontsize=9)
axes[1].set_ylim(0, 1.05)
for i, v in enumerate(r2s):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('plot_model_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance (Random Forest classifier)
rf_pipe = clf_results['Random Forest']['pipeline']
rf_model = rf_pipe.named_steps['clf']
feature_names_out = (
    numeric_cols_clf_eng +
    list(rf_pipe.named_steps['pre']
         .named_transformers_['cat']
         .named_steps['onehot']
         .get_feature_names_out(categorical_cols))
)
importances = pd.Series(rf_model.feature_importances_, index=feature_names_out)
top20 = importances.nlargest(20).sort_values()

plt.figure(figsize=(9, 6))
top20.plot(kind='barh', color='#3b82d4')
plt.title('Top 20 Feature Importances (Random Forest)', fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('plot_feature_importance.png', bbox_inches='tight')
plt.show()

---
## 8. Save Model Artifacts <a id='8'></a>

In [ ]:
os.makedirs('models', exist_ok=True)

# Save best classifier
joblib.dump(clf_results[best_clf_name]['pipeline'], 'models/placement_classifier.pkl')

# Save best regressor
joblib.dump(reg_results[best_reg_name]['pipeline'], 'models/salary_regressor.pkl')

# Save column metadata
import json
meta = {
    'numeric_cols': numeric_cols_clf_eng,
    'categorical_cols': categorical_cols,
    'best_classifier': best_clf_name,
    'best_regressor': best_reg_name,
    'classifier_accuracy': round(best_clf['accuracy'], 4),
    'classifier_auc': round(best_clf['auc'], 4),
    'regressor_r2': round(best_reg['R2'], 4)
}
with open('models/model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Models saved to /models/ ✓')
print(json.dumps(meta, indent=2))

---
## 9. Streamlit Frontend App (Single File) <a id='9'></a>

The cell below writes **`app.py`** — a complete, self-contained Streamlit application that includes:
- **Dashboard** — dataset overview, EDA charts
- **Predict** — interactive form → Placement prediction + Salary estimate
- **Model Info** — accuracy, AUC, feature importance

Run it with: `streamlit run app.py`

In [ ]:
# app.py already exists in this project directory.
# To launch the Streamlit frontend, run in your terminal:
#   streamlit run app.py

import os
app_exists = os.path.isfile("app.py")
print("app.py found:", app_exists)
print("Launch: streamlit run app.py")


---
## ✅ Project Complete!

| Artifact | Description |
|----------|-------------|
| `app.py` | Single-file Streamlit app (backend ML + frontend UI) |
| `models/placement_classifier.pkl` | Trained placement classifier |
| `models/salary_regressor.pkl` | Trained salary regressor |
| `models/model_meta.json` | Model metadata and metrics |
| `plot_*.png` | EDA and evaluation plots |

**To launch the app:** `streamlit run app.py`